<a href="https://colab.research.google.com/github/manuellazarte/Procesamiento-del-Lenguaje-Natural/blob/main/src/Trabajo_pr%C3%A1ctico_N2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install pgvector

KeyboardInterrupt: 

In [4]:
import pandas as pd

In [5]:
df = pd.read_csv("/content/libros.csv")

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   book_id           150 non-null    int64 
 1   titulo            150 non-null    object
 2   autores           150 non-null    object
 3   generos           150 non-null    object
 4   serie             48 non-null     object
 5   sinopsis          150 non-null    object
 6   url_libro         150 non-null    object
 7   categoria_origen  150 non-null    object
 8   fecha_extraccion  150 non-null    object
 9   url_portada       150 non-null    object
dtypes: int64(1), object(9)
memory usage: 11.8+ KB


In [11]:
df.head()

,book_id,titulo,autores,generos,serie,sinopsis,url_libro,categoria_origen,fecha_extraccion,url_portada
0,125386,Batallón disciplinario,['Karl von Vereiter'],"['Aventuras', 'Bélico', 'Novela']",NaN,de todas las unidades especiales que la wehrma...,/book/batallon-disciplinario/,belico,2026-09-07,https://assets.lectulandia.com/b/ab/Karl%20von...
1,125355,Bombardero,['Len Deighton'],"['Bélico', 'Intriga', 'Novela']",NaN,"en una calurosa noche de junio de 1943, una fu...",/book/bombardero/,belico,2026-09-07,https://assets.lectulandia.com/b/ab/Len%20Deig...
2,125294,Las quimeras negras,['Jean Lartéguy'],"['Bélico', 'Novela']",NaN,en medio de un áfrica presa de todos los delir...,/book/las-quimeras-negras/,belico,2026-09-07,https://assets.lectulandia.com/b/ab/Jean%20Lar...
3,125082,Tierra de asilo,['Jean Lartéguy'],"['Bélico', 'Novela']",NaN,sauveterre es un rincón de la francia olvidada...,/book/tierra-de-asilo/,belico,2026-09-07,https://assets.lectulandia.com/b/ab/Jean%20Lar...
4,124552,Colina 112,['Adrian Goldsworthy'],"['Bélico', 'Histórico', 'Novela']",NaN,"se avecina el día d. sin embargo, para tres jó...",/book/colina-112/,belico,2026-09-07,https://assets.lectulandia.com/b/ab/Adrian%20G...


In [9]:
df['sinopsis'] = df['sinopsis'].apply(lambda x: x.lower())
df['sinopsis']

,sinopsis
0,de todas las unidades especiales que la wehrma...
1,"en una calurosa noche de junio de 1943, una fu..."
2,en medio de un áfrica presa de todos los delir...
3,sauveterre es un rincón de la francia olvidada...
4,"se avecina el día d. sin embargo, para tres jó..."
...,...
145,alfa ndiaye es senegalés y ha acabado combatie...
146,veinte años después de la derrota del ejército...
147,"un día, una niña de apenas cinco años preguntó..."
148,en esta sensacional novela sobre pilotos naval...


In [14]:
df['sinopsis'] = df['sinopsis'].str.replace(r'[^\w\s]', '', regex=True)
df['sinopsis']

,sinopsis
0,de todas las unidades especiales que la wehrma...
1,en una calurosa noche de junio de 1943 una fue...
2,en medio de un áfrica presa de todos los delir...
3,sauveterre es un rincón de la francia olvidada...
4,se avecina el día d sin embargo para tres jóve...
...,...
145,alfa ndiaye es senegalés y ha acabado combatie...
146,veinte años después de la derrota del ejército...
147,un día una niña de apenas cinco años preguntó ...
148,en esta sensacional novela sobre pilotos naval...


In [15]:
import unicodedata

def remove_accents(input_str):
    nfkd_form = unicodedata.normalize('NFKD', input_str)
    return ''.join([c for c in nfkd_form if not unicodedata.combining(c)])

In [17]:
df['sinopsis'] = df['sinopsis'].apply(remove_accents)
print(df['sinopsis'])

0      de todas las unidades especiales que la wehrma...
1      en una calurosa noche de junio de 1943 una fue...
2      en medio de un africa presa de todos los delir...
3      sauveterre es un rincon de la francia olvidada...
4      se avecina el dia d sin embargo para tres jove...
                             ...                        
145    alfa ndiaye es senegales y ha acabado combatie...
146    veinte anos despues de la derrota del ejercito...
147    un dia una nina de apenas cinco anos pregunto ...
148    en esta sensacional novela sobre pilotos naval...
149    en la madrugada del 30 de abril de 1945 el rad...
Name: sinopsis, Length: 150, dtype: object


In [18]:
# Definimos la lista de stopwords en español
stopwords_es = [
    'de', 'la', 'que', 'el', 'en', 'y', 'a', 'los', 'del', 'se', 'las', 'por', 'un',
    'para', 'con', 'no', 'una', 'su', 'al', 'lo', 'como', 'más', 'pero', 'sus', 'le',
    'ya', 'o', 'este', 'sí', 'porque', 'esta', 'entre', 'cuando', 'muy', 'sin',
    'sobre', 'también', 'me', 'hasta', 'hay', 'donde', 'quien', 'desde', 'todo',
    'nos', 'durante', 'todos', 'uno', 'les', 'ni', 'contra', 'otros', 'ese', 'eso',
    'ante', 'ellos', 'e', 'esto', 'mí', 'antes', 'algunos', 'qué', 'unos', 'yo',
    'otro', 'otras', 'otra', 'él', 'tanto', 'esa', 'estos', 'mucho', 'quienes',
    'nada', 'muchos', 'cual', 'poco', 'ella', 'estar', 'estas', 'algunas', 'algo',
    'nosotros', 'mi', 'mis', 'tú', 'te', 'ti', 'tu', 'tus', 'ellas', 'nosotras',
    'vosotros', 'vosotras', 'os', 'mío', 'mía', 'míos', 'mías', 'tuyo', 'tuya',
    'tuyos', 'tuyas', 'suyo', 'suya', 'suyos', 'suyas', 'nuestro', 'nuestra',
    'nuestros', 'nuestras', 'vuestro', 'vuestra', 'vuestros', 'vuestras', 'es'
]

# Función para eliminar stopwords de un texto
def eliminar_stopwords(texto):
    # Separamos el texto en palabras
    palabras = texto.split()
    # Filtramos las palabras para eliminar las stopwords
    palabras_filtradas = [palabra for palabra in palabras if palabra.lower() not in stopwords_es]
    # Unimos las palabras filtradas en una cadena y la retornamos
    return ' '.join(palabras_filtradas)


df['sinopsis'] = df['sinopsis'].apply(eliminar_stopwords)


print("Texto sin stopwords:", df['sinopsis'])

Texto sin stopwords: 0      todas unidades especiales wehrmacht poseyo tra...
1      calurosa noche junio 1943 fuerza 700 bombarder...
2      medio africa presa delirios tres hombres consu...
3      sauveterre rincon francia olvidada cerril auto...
4      avecina dia d embargo tres jovenes gales sur g...
                             ...                        
145    alfa ndiaye senegales ha acabado combatiendo e...
146    veinte anos despues derrota ejercito italiano ...
147    dia nina apenas cinco anos pregunto hermana or...
148    sensacional novela pilotos navales guerra viet...
149    madrugada 30 abril 1945 radar ruso detecto pre...
Name: sinopsis, Length: 150, dtype: object
